# Debugging a Model That Is Not Learning

Practice the overfit-one-batch test, gradient inspection, and a common label-shuffle failure.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
print(device)

In [ ]:
class TinyMLP(nn.Module):
    def __init__(self, d=8, c=3):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, 64), nn.ReLU(), nn.Linear(64, c))
    def forward(self, x):
        return self.net(x)

def grad_norm(model):
    total = torch.tensor(0.0, device=device)
    for p in model.parameters():
        if p.grad is not None:
            total += p.grad.detach().pow(2).sum()
    return total.sqrt()

In [ ]:
# Overfit-one-batch should work.
B, D, C = 16, 8, 3
x = torch.randn(B, D, device=device)
y = torch.randint(0, C, (B,), device=device)
model = TinyMLP(D, C).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-2)

for step in range(201):
    opt.zero_grad()
    logits = model(x)
    loss = F.cross_entropy(logits, y)
    loss.backward()
    g = grad_norm(model)
    opt.step()
    if step % 50 == 0:
        acc = (logits.argmax(-1) == y).float().mean()
        print(step, 'loss', float(loss), 'acc', float(acc), 'grad_norm', float(g))

In [ ]:
# Label-shuffle bug demo: train on x with random labels each step. It should not learn.
model = TinyMLP(D, C).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
for step in range(101):
    y_bad = torch.randint(0, C, (B,), device=device)
    opt.zero_grad()
    logits = model(x)
    loss = F.cross_entropy(logits, y_bad)
    loss.backward(); opt.step()
    if step % 25 == 0:
        print(step, 'loss with changing labels', float(loss))

Interview drill: if the first overfit-one-batch cell fails, list five causes before changing architecture.